# He_MgSiO3 KSPACING convergence analysis

This notebook is the single working analysis file for the static and `v1_i1` ZONE-frame KSPACING tests.

Current interpretation from completed runs:

- Static `0.25`, `0.30`, and `0.40` are numerically indistinguishable at the current precision for the 360-atom cases; `0.50` remains small but is gamma-only for Mg-rich 360-atom cells.
- ZONE-frame tests are complete for `0.30`, `0.40`, and `0.50`; all have valid final `OSZICAR` `F=` lines.
- ZONE-frame `0.20` A100 jobs are not valid references because they fake-completed with empty `OSZICAR` files.
- Static H200 `0.20` jobs completed successfully and are now included as the densest static reference.


In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re
import statistics

import matplotlib.pyplot as plt


ROOT = Path("/work/nvme/bguf/akashgpt/qmd_data/He_MgSiO3/sim_data_ML/setup_MLMD/KSPACING_test")
FRAME_ROOT = ROOT / "v1_i1_frame_tests"

KSPACING_PATTERN = re.compile(r"KSPACING_([0-9]+)p([0-9]+)(?:_H200)?$")
TOTEN_PATTERN = re.compile(r"free\s+energy\s+TOTEN\s+=\s+([-+0-9.Ee]+)")
INTERNAL_PATTERN = re.compile(r"energy\s+without entropy=\s+([-+0-9.Ee]+)")
SIGMA0_PATTERN = re.compile(r"energy\(sigma->0\)\s+=\s+([-+0-9.Ee]+)")
NKPTS_PATTERN = re.compile(r"NKPTS\s*=\s*(\d+)")


def parse_kspacing(run_dir: Path) -> float | None:
	"""Parse KSPACING from a run directory name.

	Args:
		run_dir: Directory such as ``KSPACING_0p40``.

	Returns:
		The KSPACING value, or ``None`` for unrelated folders.
	"""
	match = KSPACING_PATTERN.match(run_dir.name)
	if match is None:
		return None
	return float(f"{int(match.group(1))}.{match.group(2)}")


def count_atoms(poscar_path: Path) -> int | None:
	"""Count atoms from a VASP POSCAR.

	Args:
		poscar_path: Path to POSCAR.

	Returns:
		Total atom count, or ``None`` if the POSCAR cannot be parsed.
	"""
	if not poscar_path.exists():
		return None
	lines = poscar_path.read_text(errors="ignore").splitlines()
	for line_index in (6, 5):
		if line_index >= len(lines):
			continue
		parts = lines[line_index].split()
		if parts and all(part.isdigit() for part in parts):
			return sum(int(part) for part in parts)
	return None


def parse_outcar(outcar_path: Path) -> dict[str, float | int | None]:
	"""Extract final energies and k-point count from OUTCAR.

	Args:
		outcar_path: Path to OUTCAR.

	Returns:
		Dictionary with final TOTEN, internal energy, sigma->0 energy, NKPTS, and line count.
	"""
	result: dict[str, float | int | None] = {
		"toten": None,
		"internal": None,
		"sigma0": None,
		"nkpts": None,
		"line_count": 0,
	}
	if not outcar_path.exists():
		return result
	lines = outcar_path.read_text(errors="ignore").splitlines()
	result["line_count"] = len(lines)
	for line in lines:
		if result["nkpts"] is None:
			match = NKPTS_PATTERN.search(line)
			if match is not None:
				result["nkpts"] = int(match.group(1))
		match = TOTEN_PATTERN.search(line)
		if match is not None:
			result["toten"] = float(match.group(1))
		match = INTERNAL_PATTERN.search(line)
		if match is not None:
			result["internal"] = float(match.group(1))
		match = SIGMA0_PATTERN.search(line)
		if match is not None:
			result["sigma0"] = float(match.group(1))
	return result


def count_final_oszicar_lines(oszicar_path: Path) -> int:
	"""Count final-energy markers in OSZICAR.

	Args:
		oszicar_path: Path to OSZICAR.

	Returns:
		Number of lines containing ``F=``.
	"""
	if not oszicar_path.exists():
		return 0
	return sum(1 for line in oszicar_path.read_text(errors="ignore").splitlines() if " F=" in line)


In [ ]:
def load_static_rows(root: Path) -> list[dict[str, object]]:
	"""Load completed static KSPACING results.

	Args:
		root: Static KSPACING test root.

	Returns:
		Rows with energies and validity flags.
	"""
	rows: list[dict[str, object]] = []
	for run_dir in root.glob("*/*/KSPACING_*"):
		if not run_dir.is_dir() or "v1_i1_frame_tests" in str(run_dir):
			continue
		kspacing = parse_kspacing(run_dir)
		if kspacing is None:
			continue
		outcar_data = parse_outcar(run_dir / "OUTCAR")
		n_atoms = count_atoms(run_dir / "POSCAR")
		final_line_count = count_final_oszicar_lines(run_dir / "OSZICAR")
		valid = bool(
			n_atoms
			and final_line_count > 0
			and outcar_data["toten"] is not None
			and outcar_data["internal"] is not None
		)
		rows.append(
			{
				"family": run_dir.parts[-3],
				"config": run_dir.parts[-2],
				"run": run_dir.name,
				"kspacing": kspacing,
				"n_atoms": n_atoms,
				"final_line_count": final_line_count,
				"valid": valid,
				"path": str(run_dir),
				**outcar_data,
			}
		)
	return rows


def load_frame_rows(frame_root: Path) -> list[dict[str, object]]:
	"""Load ZONE-frame KSPACING results from the frame-test manifest.

	Args:
		frame_root: Frame-test root containing ``submitted_jobs.tsv``.

	Returns:
		Rows with energies and validity flags.
	"""
	rows: list[dict[str, object]] = []
	with (frame_root / "submitted_jobs.tsv").open(encoding="utf-8") as handle:
		for manifest_row in csv.DictReader(handle, delimiter="\t"):
+			run_dir = Path(manifest_row["run_dir"])
+			outcar_data = parse_outcar(run_dir / "OUTCAR")
+			n_atoms = count_atoms(run_dir / "POSCAR")
+			final_line_count = count_final_oszicar_lines(run_dir / "OSZICAR")
+			valid = bool(
+				n_atoms
+				and final_line_count > 0
+				and outcar_data["toten"] is not None
+				and outcar_data["internal"] is not None
+			)
+			rows.append(
+				{
+					**manifest_row,
+					"kspacing": float(manifest_row["kspacing"]),
+					"n_atoms": n_atoms,
+					"final_line_count": final_line_count,
+					"valid": valid,
+					"path": str(run_dir),
+					**outcar_data,
+				}
+			)
+	return rows


static_rows = load_static_rows(ROOT)
frame_rows = load_frame_rows(FRAME_ROOT)

valid_static_rows = [row for row in static_rows if row["valid"]]
valid_frame_rows = [row for row in frame_rows if row["valid"]]

print(f"Static valid rows: {len(valid_static_rows)} / {len(static_rows)}")
print(f"Frame valid rows: {len(valid_frame_rows)} / {len(frame_rows)}")
print(f"Invalid frame 0.20 rows: {sum((not row['valid']) and row['kspacing'] == 0.20 for row in frame_rows)}")


In [ ]:
def energy_delta_mev_per_atom(row: dict[str, object], reference: dict[str, object], energy_key: str) -> float:
	"""Compute absolute energy difference in meV/atom.

	Args:
		row: Target result row.
		reference: Reference result row.
		energy_key: Energy key, usually ``toten`` or ``internal``.

	Returns:
		Absolute energy difference in meV/atom.
	"""
	return abs((float(row[energy_key]) - float(reference[energy_key])) / int(row["n_atoms"]) * 1000.0)


static_by_config: dict[tuple[str, str], list[dict[str, object]]] = defaultdict(list)
for row in valid_static_rows:
	static_by_config[(str(row["family"]), str(row["config"]))].append(row)

print("Static deltas vs smallest valid KSPACING per composition")
for config_key, rows in sorted(static_by_config.items()):
	reference = min(rows, key=lambda item: float(item["kspacing"]))
	print(f"\n{config_key}; reference={reference['run']} (K={reference['kspacing']:.2f}, NKPTS={reference['nkpts']})")
	for row in sorted(rows, key=lambda item: float(item["kspacing"])):
		print(
			f"K={row['kspacing']:.2f}, NKPTS={row['nkpts']}, "
			f"|dTOTEN|={energy_delta_mev_per_atom(row, reference, 'toten'):.6f} meV/atom, "
			f"|dInternal|={energy_delta_mev_per_atom(row, reference, 'internal'):.6f} meV/atom"
		)


In [ ]:
frame_by_frame: dict[tuple[str, str, str], list[dict[str, object]]] = defaultdict(list)
for row in valid_frame_rows:
	frame_by_frame[(str(row["zone"]), str(row["config"]), str(row["frame"]))].append(row)

frame_delta_rows: list[dict[str, object]] = []
print("ZONE-frame deltas vs KSPACING=0.30 for each frame")
for frame_key, rows in sorted(frame_by_frame.items()):
	reference_options = [row for row in rows if math.isclose(float(row["kspacing"]), 0.30)]
	reference = reference_options[0] if reference_options else min(rows, key=lambda item: float(item["kspacing"]))
	print(f"\n{frame_key}; reference K={reference['kspacing']:.2f}, NKPTS={reference['nkpts']}")
	for row in sorted(rows, key=lambda item: float(item["kspacing"])):
		delta_toten = energy_delta_mev_per_atom(row, reference, "toten")
+		delta_internal = energy_delta_mev_per_atom(row, reference, "internal")
+		frame_delta_rows.append(
+			{
+				"zone": row["zone"],
+				"config": row["config"],
+				"frame": row["frame"],
+				"kspacing": row["kspacing"],
+				"nkpts": row["nkpts"],
+				"delta_toten_mev_atom": delta_toten,
+				"delta_internal_mev_atom": delta_internal,
+			}
+		)
+		print(
+			f"K={row['kspacing']:.2f}, NKPTS={row['nkpts']}, "
+			f"|dTOTEN|={delta_toten:.6f} meV/atom, "
+			f"|dInternal|={delta_internal:.6f} meV/atom"
+		)
+

In [ ]:
print("Frame aggregate convergence vs KSPACING=0.30")
+for kspacing in sorted({float(row["kspacing"]) for row in frame_delta_rows}):
+	rows = [row for row in frame_delta_rows if math.isclose(float(row["kspacing"]), kspacing)]
+	delta_toten_values = [float(row["delta_toten_mev_atom"]) for row in rows]
+	delta_internal_values = [float(row["delta_internal_mev_atom"]) for row in rows]
+	print(
+		f"K={kspacing:.2f}, n={len(rows)}, "
+		f"max |dTOTEN|={max(delta_toten_values):.6f}, "
+		f"mean |dTOTEN|={statistics.mean(delta_toten_values):.6f}, "
+		f"max |dInternal|={max(delta_internal_values):.6f}, "
+		f"mean |dInternal|={statistics.mean(delta_internal_values):.6f} meV/atom"
+	)
+
+print("\nWorst frame internal-energy deltas")
+for row in sorted(frame_delta_rows, key=lambda item: float(item["delta_internal_mev_atom"]), reverse=True)[:8]:
+	print(
+		f"K={row['kspacing']:.2f}, |dInternal|={row['delta_internal_mev_atom']:.6f} meV/atom, "
+		f"NKPTS={row['nkpts']}, {row['zone']} {row['config']} {row['frame']}"
+	)
+

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
+
+for config_key, rows in sorted(static_by_config.items()):
+	reference = min(rows, key=lambda item: float(item["kspacing"]))
+	x_values = [float(row["kspacing"]) for row in sorted(rows, key=lambda item: float(item["kspacing"]))]
+	y_values = [energy_delta_mev_per_atom(row, reference, "internal") for row in sorted(rows, key=lambda item: float(item["kspacing"]))]
+	axes[0].plot(x_values, y_values, marker="o", linewidth=1.5, label=" / ".join(config_key))
+
+for config in sorted({str(row["config"]) for row in frame_delta_rows}):
+	for zone in sorted({str(row["zone"]) for row in frame_delta_rows if str(row["config"]) == config}):
+		rows = [row for row in frame_delta_rows if str(row["config"]) == config and str(row["zone"]) == zone]
+		aggregated: dict[float, list[float]] = defaultdict(list)
+		for row in rows:
+			aggregated[float(row["kspacing"])].append(float(row["delta_internal_mev_atom"]))
+		x_values = sorted(aggregated)
+		y_values = [max(aggregated[kspacing]) for kspacing in x_values]
+		axes[1].plot(x_values, y_values, marker="o", linewidth=1.5, label=f"{zone} {config}")
+
+axes[0].set_title("Static tests")
+axes[0].set_xlabel("KSPACING")
+axes[0].set_ylabel("|Δ internal energy| (meV/atom)")
+axes[0].grid(True, alpha=0.25)
+axes[0].legend(fontsize=7)
+
+axes[1].set_title("ZONE-frame tests, max over two frames")
+axes[1].set_xlabel("KSPACING")
+axes[1].set_ylabel("|Δ internal energy| vs K=0.30 (meV/atom)")
+axes[1].grid(True, alpha=0.25)
+axes[1].legend(fontsize=7)
+
+plt.show()
+

## Working conclusion

For the completed data available now, `KSPACING=0.40` is the safe recommendation.

Why:

- Static tests show `0.40` is effectively converged relative to the valid H200 `0.20` reference.
- ZONE-frame tests show `0.40` has worst-case internal-energy deviation of about `0.154 meV/atom` relative to `0.30`, with most cases far smaller.
- `0.50` is also very close energetically, but it becomes gamma-only in the Mg-rich larger cells and therefore feels less conservative for production.
- `0.20` should only be used as a confirmatory reference on H200/CPU, not as the practical default, because it is too expensive and failed on A100 memory for the large cases.


In [ ]:
"""Plot static KSPACING convergence with k-point mesh and runtime panels."""

from __future__ import annotations

import csv
import os
import re
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


BENCHMARK_DIR = Path(os.environ.get("KSPACING_BENCHMARK_DIR", Path(__file__).resolve().parent))
INPUT_CSV = BENCHMARK_DIR / "kspacing_convergence_delta_mev_per_atom_static.csv"
OUTPUT_PNG = Path(
	os.environ.get(
		"KSPACING_OUTPUT_PNG",
		BENCHMARK_DIR / "kspacing_convergence_delta_mev_per_atom_static.png",
	)
)

KPOINT_MESH_PATTERN = re.compile(r"generate k-points for:\s*(?P<nkx>\d+)\s+(?P<nky>\d+)\s+(?P<nkz>\d+)")
ELAPSED_PATTERN = re.compile(r"Elapsed time \(sec\):\s*(?P<seconds>[0-9.]+)")
RESOURCE_PATTERN = re.compile(
	r"running\s+(?P<mpi_ranks>\d+)\s+mpi-ranks,\s+with\s+(?P<threads>\d+)\s+threads/rank"
)
GPU_PATTERN = re.compile(r"Offloading initialized\s+\.\.\.\s+(?P<gpus>\d+)\s+GPUs detected")
PARTITION_PATTERN = re.compile(r"#SBATCH\s+--partition=(?P<partition>\S+)")
ACCELERATOR_MARKERS = {"A100": "^", "H200": "*", "unknown": "o"}


@dataclass(frozen=True)
class StaticKspacingRow:
	"""One completed static KSPACING convergence calculation."""

	family: str
	config: str
	run: str
	kspacing: float
	nkpts: int
	delta_toten_mev_per_atom: float
	delta_internal_mev_per_atom: float
	kpoint_mesh: tuple[int, int, int]
	elapsed_seconds: float
	mpi_ranks: int | None
	threads_per_rank: int | None
	gpus_detected: int | None
	accelerator: str

	@property
	def label(self) -> str:
		"""Return the compact legend label used in the convergence plots."""
		family_label = self.family.replace("_atoms__and_He", " atoms")
		return f"{family_label}, {self.config}"


def read_outcar_metadata(outcar_path: Path) -> tuple[tuple[int, int, int], float, int | None, int | None, int | None]:
	"""Read VASP mesh, elapsed time, and resource metadata from OUTCAR."""
	kpoint_mesh: tuple[int, int, int] | None = None
	elapsed_seconds: float | None = None
	mpi_ranks: int | None = None
	threads_per_rank: int | None = None
	gpus_detected: int | None = None
	with outcar_path.open("r", encoding="utf-8", errors="replace") as handle:
		for line in handle:
			if kpoint_mesh is None and (match := KPOINT_MESH_PATTERN.search(line)):
				kpoint_mesh = (
					int(match.group("nkx")),
					int(match.group("nky")),
					int(match.group("nkz")),
				)
			if elapsed_seconds is None and (match := ELAPSED_PATTERN.search(line)):
				elapsed_seconds = float(match.group("seconds"))
			if mpi_ranks is None and (match := RESOURCE_PATTERN.search(line)):
				mpi_ranks = int(match.group("mpi_ranks"))
				threads_per_rank = int(match.group("threads"))
			if gpus_detected is None and (match := GPU_PATTERN.search(line)):
				gpus_detected = int(match.group("gpus"))
	if kpoint_mesh is None:
		raise ValueError(f"No k-point mesh line found in {outcar_path}")
	if elapsed_seconds is None:
		raise ValueError(f"No elapsed-time line found in {outcar_path}")
	return kpoint_mesh, elapsed_seconds, mpi_ranks, threads_per_rank, gpus_detected


def read_accelerator(run_path: Path) -> str:
	"""Read accelerator class from the local submission script."""
	for script_path in sorted(run_path.glob("sub*.sh")):
		with script_path.open("r", encoding="utf-8", errors="replace") as handle:
			for line in handle:
				match = PARTITION_PATTERN.search(line)
				if match is None:
					continue
				partition = match.group("partition")
				if "H200" in partition:
					return "H200"
				if "A100" in partition:
					return "A100"
	return "unknown"


def load_static_kspacing_rows(csv_path: Path) -> list[StaticKspacingRow]:
	"""Load static KSPACING convergence rows from the saved CSV."""
	rows: list[StaticKspacingRow] = []
	with csv_path.open("r", encoding="utf-8", newline="") as handle:
		for raw_row in csv.DictReader(handle):
			run_path = BENCHMARK_DIR / raw_row["family"] / raw_row["config"] / raw_row["run"]
			kpoint_mesh, elapsed_seconds, mpi_ranks, threads_per_rank, gpus_detected = read_outcar_metadata(
				run_path / "OUTCAR"
			)
			rows.append(
				StaticKspacingRow(
					family=raw_row["family"],
					config=raw_row["config"],
					run=raw_row["run"],
					kspacing=float(raw_row["kspacing"]),
					nkpts=int(raw_row["nkpts"]),
					delta_toten_mev_per_atom=float(raw_row["delta_toten_meV_per_atom"]),
					delta_internal_mev_per_atom=float(raw_row["delta_internal_meV_per_atom"]),
					kpoint_mesh=kpoint_mesh,
					elapsed_seconds=elapsed_seconds,
					mpi_ranks=mpi_ranks,
					threads_per_rank=threads_per_rank,
					gpus_detected=gpus_detected,
					accelerator=read_accelerator(run_path),
				)
			)
	return sorted(rows, key=lambda item: (item.family, item.config, -item.kspacing))


def group_static_rows(rows: Iterable[StaticKspacingRow]) -> OrderedDict[tuple[str, str], list[StaticKspacingRow]]:
	"""Group static rows by simulation family and composition."""
	grouped_rows: OrderedDict[tuple[str, str], list[StaticKspacingRow]] = OrderedDict()
	for row in rows:
		grouped_rows.setdefault((row.family, row.config), []).append(row)
	return grouped_rows


def configure_energy_axis(axis: plt.Axes, title: str) -> None:
	"""Apply shared styling to a static convergence energy axis."""
	axis.set_title(title, fontsize=14)
	axis.set_xlabel("KSPACING", fontsize=11)
	axis.set_ylabel("meV/atom", fontsize=11)
	axis.set_yscale("symlog", linthresh=1.0e-5)
	axis.set_ylim(bottom=0.0)
	axis.grid(True, which="both", alpha=0.25)
	axis.tick_params(axis="both", labelsize=10)
	axis.invert_xaxis()


def add_energy_panels(
	axes: tuple[plt.Axes, plt.Axes],
	grouped_rows: OrderedDict[tuple[str, str], list[StaticKspacingRow]],
) -> None:
	"""Add the two top static KSPACING convergence panels."""
	toten_axis, internal_axis = axes
	for rows in grouped_rows.values():
		ordered_rows = sorted(rows, key=lambda item: item.kspacing, reverse=True)
		kspacing_values = [row.kspacing for row in ordered_rows]
		label = ordered_rows[0].label
		toten_axis.plot(
			kspacing_values,
			[row.delta_toten_mev_per_atom for row in ordered_rows],
			marker="o",
			linewidth=1.8,
			markersize=6.5,
			label=label,
		)
		internal_axis.plot(
			kspacing_values,
			[row.delta_internal_mev_per_atom for row in ordered_rows],
			marker="s",
			linewidth=1.8,
			markersize=6.5,
			label=label,
		)
	configure_energy_axis(toten_axis, r"$|\Delta$ TOTEN| vs smallest valid KSPACING")
	configure_energy_axis(internal_axis, r"$|\Delta$ internal energy| vs smallest valid KSPACING")
	toten_axis.legend(fontsize=8, loc="upper right")
	internal_axis.legend(fontsize=8, loc="upper right")


def add_kpoint_panel(
	axis: plt.Axes,
	grouped_rows: OrderedDict[tuple[str, str], list[StaticKspacingRow]],
) -> None:
	"""Add the k-point mesh panel with staggered overlapping points."""
	unique_meshes = sorted(
		{row.kpoint_mesh for rows in grouped_rows.values() for row in rows},
		key=lambda mesh: (mesh[0] * mesh[1] * mesh[2], mesh),
	)
	mesh_positions = {mesh: index for index, mesh in enumerate(unique_meshes)}
	mesh_labels = [f"{mesh[0]}x{mesh[1]}x{mesh[2]}" for mesh in unique_meshes]
	mesh_nkpts = [mesh[0] * mesh[1] * mesh[2] for mesh in unique_meshes]
	collision_counts: dict[tuple[float, int], int] = {}
	collision_ranks: dict[tuple[float, int], int] = {}
	for rows in grouped_rows.values():
		for row in rows:
			key = (row.kspacing, mesh_positions[row.kpoint_mesh])
			collision_counts[key] = collision_counts.get(key, 0) + 1
	for rows in grouped_rows.values():
		ordered_rows = sorted(rows, key=lambda item: item.kspacing, reverse=True)
		kspacing_values = [row.kspacing for row in ordered_rows]
		positions: list[float] = []
		for row in ordered_rows:
			base_position = mesh_positions[row.kpoint_mesh]
			key = (row.kspacing, base_position)
			rank = collision_ranks.get(key, 0)
			collision_ranks[key] = rank + 1
			count = collision_counts[key]
			offset = (rank - (count - 1) / 2.0) * 0.12 if count > 1 else 0.0
			positions.append(base_position + offset)
		line = axis.plot(kspacing_values, positions, linewidth=1.3, alpha=0.55, label=ordered_rows[0].label)[0]
		axis.scatter(
			kspacing_values,
			positions,
			s=115,
			color=line.get_color(),
			edgecolor="black",
			linewidth=0.85,
			zorder=3,
		)
	axis.set_title("VASP-generated k-point mesh", fontsize=12)
	axis.set_xlabel("KSPACING", fontsize=11)
	axis.set_ylabel("k-point mesh", fontsize=10)
	axis.set_yticks(range(len(mesh_labels)))
	axis.set_yticklabels(mesh_labels, fontsize=9)
	axis.set_ylim(-0.45, len(mesh_labels) - 0.55)
	axis.grid(True, alpha=0.25)
	axis.legend(fontsize=8, loc="upper left")
	axis.tick_params(axis="x", labelsize=10)
	axis.invert_xaxis()
	nkpts_axis = axis.twinx()
	nkpts_axis.set_ylim(axis.get_ylim())
	nkpts_axis.set_yticks(range(len(mesh_nkpts)))
	nkpts_axis.set_yticklabels([str(nkpts) for nkpts in mesh_nkpts], fontsize=9)
	nkpts_axis.set_ylabel("NKPTS", fontsize=10)


def describe_resources(rows: Iterable[StaticKspacingRow]) -> str:
	"""Describe the common detected resource footprint for plot labels."""
	resource_keys = {
		(row.mpi_ranks, row.threads_per_rank, row.gpus_detected)
		for row in rows
		if row.mpi_ranks is not None or row.threads_per_rank is not None or row.gpus_detected is not None
	}
	if len(resource_keys) != 1:
		return "detected resources vary"
	mpi_ranks, threads_per_rank, gpus_detected = next(iter(resource_keys))
	parts: list[str] = []
	if mpi_ranks is not None and threads_per_rank is not None:
		parts.append(f"{mpi_ranks} MPI rank x {threads_per_rank} thread")
	if gpus_detected is not None:
		parts.append(f"{gpus_detected} GPU")
	accelerators = sorted({row.accelerator for row in rows})
	if len(accelerators) == 1 and accelerators[0] != "unknown":
		parts.append(accelerators[0])
	elif any(accelerator != "unknown" for accelerator in accelerators):
		parts.append("mixed A100/H200")
	return ", ".join(parts) if parts else "detected resources unavailable"


def add_runtime_panel(
	axis: plt.Axes,
	grouped_rows: OrderedDict[tuple[str, str], list[StaticKspacingRow]],
) -> None:
	"""Add the runtime panel using VASP elapsed wall time."""
	for rows in grouped_rows.values():
		ordered_rows = sorted(rows, key=lambda item: item.kspacing, reverse=True)
		line = axis.plot(
			[row.kspacing for row in ordered_rows],
			[row.elapsed_seconds / 60.0 for row in ordered_rows],
			linewidth=1.7,
			label=ordered_rows[0].label,
		)[0]
		for row in ordered_rows:
			marker = ACCELERATOR_MARKERS.get(row.accelerator, "o")
			axis.scatter(
				row.kspacing,
				row.elapsed_seconds / 60.0,
				marker=marker,
				s=92 if marker != "*" else 150,
				color=line.get_color(),
				edgecolor="black",
				linewidth=0.8,
				zorder=3,
			)
			if row.accelerator == "H200":
				axis.annotate(
					"H200",
					(row.kspacing, row.elapsed_seconds / 60.0),
					textcoords="offset points",
					xytext=(5, 6),
					fontsize=7,
				)
	all_rows = [row for rows in grouped_rows.values() for row in rows]
	axis.set_title(f"VASP elapsed runtime ({describe_resources(all_rows)})", fontsize=12)
	axis.set_xlabel("KSPACING", fontsize=11)
	axis.set_ylabel("elapsed time (min)", fontsize=10)
	axis.grid(True, alpha=0.25)
	series_legend = axis.legend(fontsize=8, loc="upper left")
	axis.add_artist(series_legend)
	accelerators = [name for name in ("A100", "H200") if any(row.accelerator == name for row in all_rows)]
	if accelerators:
		axis.legend(
			[
				Line2D([0], [0], marker=ACCELERATOR_MARKERS[name], color="none", markerfacecolor="white", markeredgecolor="black", markersize=8)
				for name in accelerators
			],
			accelerators,
			fontsize=8,
			loc="upper right",
			title="GPU",
			title_fontsize=8,
		)
	axis.tick_params(axis="both", labelsize=10)
	axis.invert_xaxis()


def plot_static_kspacing_convergence(rows: list[StaticKspacingRow], output_png: Path) -> None:
	"""Create the static KSPACING convergence PNG."""
	grouped_rows = group_static_rows(rows)
	fig = plt.figure(figsize=(13.8, 10.4), dpi=220, constrained_layout=True)
	grid_spec = fig.add_gridspec(3, 2, height_ratios=[1.0, 0.58, 0.58])
	toten_axis = fig.add_subplot(grid_spec[0, 0])
	internal_axis = fig.add_subplot(grid_spec[0, 1])
	kpoint_axis = fig.add_subplot(grid_spec[1, :])
	runtime_axis = fig.add_subplot(grid_spec[2, :])
	add_energy_panels((toten_axis, internal_axis), grouped_rows)
	add_kpoint_panel(kpoint_axis, grouped_rows)
	add_runtime_panel(runtime_axis, grouped_rows)
	fig.suptitle("Static KSPACING convergence including H200 K=0.20", fontsize=16)
	fig.savefig(output_png)
	plt.close(fig)


def main() -> None:
	"""Run the plotting workflow."""
	static_kspacing_rows = load_static_kspacing_rows(INPUT_CSV)
	plot_static_kspacing_convergence(static_kspacing_rows, OUTPUT_PNG)
	print(f"Wrote {OUTPUT_PNG}")


if __name__ == "__main__":
	main()
